In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive_output, FloatSlider, RadioButtons, HBox, VBox, Layout, HTML, Output
from IPython.display import display

# ------------------------------------------------------------
# 1. DESCRIPTION
# ------------------------------------------------------------

description = HTML("""
<div style="
    border:1px solid #b9d7f5;
    border-radius:8px;
    padding:8px 10px;
    margin-bottom:10px;
    font-size:13px;
    line-height:1.35;
    background-color:#f7fbff;
">
<div><b>Purpose:</b> Explore the component-level behavior of second-order Sallen-Key active filters.</div>
<div><b>Available filters:</b> Low-pass, high-pass, and band-pass.</div>
<div><b>What we see:</b> The magnitude and phase responses together with the values of the gain, ω₀, and Q calculated directly from the circuit components.</div>
<div><b>What happens as we interact:</b> Changing the resistors and capacitors modifies the filter parameters and immediately changes its frequency response.</div>
<div style="margin-top:5px;"><b>Note:</b> The Sallen-Key band-stop topology is not implemented because it is rarely used in practice due to its undesirable tuning properties.</div>
</div>
""")

# ------------------------------------------------------------
# 2. FILTER TYPE
# ------------------------------------------------------------

filter_selector = RadioButtons(options=['Low-pass', 'High-pass', 'Band-pass'], value='Low-pass', description='', layout=Layout(width='360px', height='26px', margin='0px', padding='0px'))

filter_selector.add_class('horizontal-radio')

radio_style = HTML("""
<style>

.horizontal-radio {
    margin:0 !important;
    padding:0 !important;
    height:26px !important;
}

.horizontal-radio .widget-radio-box {
    display:flex !important;
    flex-direction:row !important;
    flex-wrap:nowrap !important;
    align-items:center !important;
    gap:24px !important;
    margin:0 !important;
    padding:0 !important;
    height:26px !important;
}

.horizontal-radio .widget-radio-box label {
    display:flex !important;
    flex-direction:row !important;
    align-items:center !important;
    margin:0 !important;
    padding:0 !important;
    height:26px !important;
    line-height:26px !important;
    white-space:nowrap !important;
    font-size:13px !important;
    font-weight:bold !important;
}

.horizontal-radio .widget-radio-box input {
    margin:0 0 0 8px !important;
    padding:0 !important;
    vertical-align:middle !important;
}

.horizontal-radio > label {
    display:none !important;
}

</style>
""")

filter_type_label = HTML("""
<div style="
    display:flex;
    align-items:center;
    justify-content:flex-start;
    height:26px;
    line-height:26px;
    margin:0 25px 0 0;
    padding:0;
    font-size:13px;
    font-weight:bold;
    white-space:nowrap;
">
Filter Type:
</div>
""")

# ------------------------------------------------------------
# 3. CIRCUIT PARAMETERS
# ------------------------------------------------------------

r1_slider = FloatSlider(min=5.0, max=30.0, step=0.5, value=10.0, description='R₁:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))

r2_slider = FloatSlider(min=5.0, max=30.0, step=0.5, value=10.0, description='R₂:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))

r3_slider = FloatSlider(min=5.0, max=30.0, step=0.5, value=10.0, description='R₃:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))

r4_slider = FloatSlider(min=0.0, max=15.0, step=0.5, value=5.0, description='R₄:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))

c1_slider = FloatSlider(min=5.0, max=50.0, step=1.0, value=10.0, description='C₁:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))

c2_slider = FloatSlider(min=5.0, max=50.0, step=1.0, value=10.0, description='C₂:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'35px'}, layout=Layout(width='220px'))

# ------------------------------------------------------------
# 4. LEGEND AND LABELS
# ------------------------------------------------------------

legend_html = HTML("""
<div style="
    border:1px solid #cccccc;
    border-radius:5px;
    padding:7px 9px;
    width:135px;
    font-size:13px;
    line-height:1.7;
    background:white;
">
<div><span style="display:inline-block; width:32px; border-top:3px solid red; vertical-align:middle; margin-right:7px;"></span>|H(jω)|</div>
<div><span style="display:inline-block; width:32px; border-top:2px dotted black; vertical-align:middle; margin-right:7px;"></span>ω₀</div>
</div>
""")

parameter_label = HTML("<div style='font-size:14px; font-weight:bold; margin-top:12px; margin-bottom:4px;'>Circuit Parameters:</div>")

units_html = HTML("""
<div style="
    border:1px solid #cccccc;
    border-radius:5px;
    padding:7px 9px;
    margin-top:7px;
    font-size:12px;
    line-height:1.6;
    color:#555555;
    background:white;
    width:110px;
">
R₁, R₂, R₃, R₄ in kΩ<br>
C₁, C₂ in nF
</div>
""")

# ------------------------------------------------------------
# 5. OUTPUT AREAS
# ------------------------------------------------------------

magnitude_output = Output(layout=Layout(width='100%', overflow='hidden'))
phase_output = Output(layout=Layout(width='100%', overflow='hidden'))
info_output = Output(layout=Layout(width='auto', overflow='hidden'))

# ------------------------------------------------------------
# 6. MAIN UPDATE FUNCTION
# ------------------------------------------------------------

def update_sallen_key(filter_type, R1, R2, R3, R4, C1, C2):

    R1_si = R1 * 1e3
    R2_si = R2 * 1e3
    R3_si = R3 * 1e3
    R4_si = R4 * 1e3
    C1_si = C1 * 1e-9
    C2_si = C2 * 1e-9

    K = 1.0 + R4_si / R3_si

    if filter_type == 'Low-pass':

        omega0_squared = 1.0 / (R1_si * R2_si * C1_si * C2_si)
        omega0 = np.sqrt(omega0_squared)
        a1 = 1.0 / (R1_si * C2_si) + 1.0 / (R2_si * C2_si) - (R4_si / R3_si) / (R2_si * C1_si)
        numerator_type = 'low'
        numerator_factor = K * omega0_squared

    elif filter_type == 'High-pass':

        omega0_squared = 1.0 / (R1_si * R2_si * C1_si * C2_si)
        omega0 = np.sqrt(omega0_squared)
        a1 = 1.0 / (R1_si * C1_si) + 1.0 / (R1_si * C2_si) - (R4_si / R3_si) / (R1_si * C1_si)
        numerator_type = 'high'
        numerator_factor = K

    else:

        omega0_squared = (R1_si + R2_si) / (R1_si * R2_si * R3_si * C1_si * C2_si)
        omega0 = np.sqrt(omega0_squared)
        a1 = 1.0 / (R1_si * C1_si) + 1.0 / (R3_si * C2_si) + 1.0 / (R3_si * C1_si) - (R4_si / R3_si) / (R2_si * C1_si)
        numerator_type = 'band'
        numerator_factor = K / (R1_si * C1_si)

    if a1 <= 0.0:

        with magnitude_output:
            magnitude_output.clear_output(wait=True)
            display(HTML("""
            <div style="
                border:1px solid #d9534f;
                border-radius:7px;
                padding:15px;
                margin-top:10px;
                font-size:14px;
                background-color:#fff7f7;
            ">
            <b>Invalid parameter combination:</b> the coefficient of s in the denominator is not positive.
            Change the circuit element values to obtain a stable second-order response.
            </div>
            """))

        with phase_output:
            phase_output.clear_output(wait=True)

        with info_output:
            info_output.clear_output(wait=True)

        return

    Q = omega0 / a1

    omega = np.logspace(np.log10(omega0 / 20.0), np.log10(omega0 * 20.0), 4000)

    denominator = (omega0_squared - omega**2) + 1j * a1 * omega

    if numerator_type == 'low':
        H = numerator_factor / denominator

    elif numerator_type == 'high':
        H = numerator_factor * (1j * omega)**2 / denominator

    else:
        H = numerator_factor * (1j * omega) / denominator

    magnitude = np.abs(H)
    phase = np.angle(H, deg=True)

    if filter_type == 'Low-pass':
        reference_gain = K
        reference_name = 'DC gain'

    elif filter_type == 'High-pass':
        reference_gain = K
        reference_name = 'High-frequency gain'

    else:
        H0 = numerator_factor * (1j * omega0) / (1j * a1 * omega0)
        reference_gain = np.abs(H0)
        reference_name = 'Peak gain'

    # --------------------------------------------------------
    # MAGNITUDE RESPONSE
    # --------------------------------------------------------

    with magnitude_output:

        magnitude_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.2, 3.5))

        ax.semilogx(omega, magnitude, 'r-', linewidth=2.0)
        ax.axvline(omega0, color='black', linestyle=':', linewidth=1.5)

        magnitude_at_omega0 = np.interp(omega0, omega, magnitude)

        ax.plot(omega0, magnitude_at_omega0, 'ko', markersize=4)

        y_max = max(3.0, min(10.0, np.max(magnitude) * 1.15))

        ax.set_xlim(omega[0], omega[-1])
        ax.set_ylim(0.0, y_max)

        ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=11)
        ax.set_ylabel('Magnitude |H(jω)|', fontsize=11)
        ax.set_title(f'Sallen-Key Second-Order {filter_type} Filter — Magnitude Response', fontsize=13, fontweight='bold', pad=8)

        ax.grid(True, which='both', linestyle=':', alpha=0.35)

        plt.tight_layout()
        plt.show()
        plt.close(fig)

    # --------------------------------------------------------
    # PHASE RESPONSE
    # --------------------------------------------------------

    with phase_output:

        phase_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.2, 3.2))

        ax.semilogx(omega, phase, 'r-', linewidth=2.0)
        ax.axvline(omega0, color='black', linestyle=':', linewidth=1.5)
        ax.axhline(0.0, color='gray', linestyle='--', linewidth=1.0)

        ax.set_xlim(omega[0], omega[-1])

        if filter_type == 'Low-pass':
            ax.set_ylim(-190.0, 10.0)
            ax.set_yticks([0, -45, -90, -135, -180])

        elif filter_type == 'High-pass':
            ax.set_ylim(-10.0, 190.0)
            ax.set_yticks([0, 45, 90, 135, 180])

        else:
            ax.set_ylim(-100.0, 100.0)
            ax.set_yticks([-90, -45, 0, 45, 90])

        ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=11)
        ax.set_ylabel('Phase (degrees)', fontsize=11)
        ax.set_title(f'Sallen-Key Second-Order {filter_type} Filter — Phase Response', fontsize=13, fontweight='bold', pad=8)

        ax.grid(True, which='both', linestyle=':', alpha=0.35)

        plt.tight_layout()
        plt.show()
        plt.close(fig)

    # --------------------------------------------------------
    # RESPONSE REGIME
    # --------------------------------------------------------

    if Q < 1.0 / np.sqrt(2.0):
        behavior = 'Q < 1/√2'

    elif np.isclose(Q, 1.0 / np.sqrt(2.0), atol=0.01):
        behavior = 'Q ≈ 1/√2'

    else:
        behavior = 'Q > 1/√2'

    # --------------------------------------------------------
    # COMPACT INFORMATION FRAME
    # --------------------------------------------------------

    info_html = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:6px 9px;
        font-size:12px;
        background:white;
        display:inline-block;
        width:auto;
        box-sizing:border-box;
        white-space:nowrap;
    ">

        <div style="display:flex; align-items:center; gap:22px;">
            <div><b>Filter:</b> <span style="color:#0066cc;">{filter_type}</span></div>
            <div><b>K:</b> <span style="color:#0066cc;">{K:.3f}</span></div>
            <div><b>ω₀:</b> <span style="color:#0066cc;">{omega0:.2f} rad/s</span></div>
            <div><b>f₀:</b> <span style="color:#0066cc;">{omega0 / (2.0 * np.pi):.2f} Hz</span></div>
            <div><b>Q:</b> <span style="color:#0066cc;">{Q:.3f}</span></div>
        </div>

        <div style="margin-top:4px; padding-top:4px; border-top:1px solid #eeeeee; display:flex; align-items:center; gap:22px;">
            <div><b>{reference_name}:</b> <span style="color:#0066cc;">{reference_gain:.3f}</span></div>
            <div><b>Regime:</b> <span style="color:#0066cc;">{behavior}</span></div>
        </div>

        <div style="margin-top:4px; padding-top:4px; border-top:1px solid #eeeeee; display:flex; align-items:center; gap:18px;">
            <div><b>R₁:</b> <span style="color:#0066cc;">{R1:.1f} kΩ</span></div>
            <div><b>R₂:</b> <span style="color:#0066cc;">{R2:.1f} kΩ</span></div>
            <div><b>R₃:</b> <span style="color:#0066cc;">{R3:.1f} kΩ</span></div>
            <div><b>R₄:</b> <span style="color:#0066cc;">{R4:.1f} kΩ</span></div>
            <div><b>C₁:</b> <span style="color:#0066cc;">{C1:.1f} nF</span></div>
            <div><b>C₂:</b> <span style="color:#0066cc;">{C2:.1f} nF</span></div>
        </div>

    </div>
    """

    with info_output:
        info_output.clear_output(wait=True)
        display(HTML(info_html))

# ------------------------------------------------------------
# 7. INTERACTION
# ------------------------------------------------------------

interactive_controls = interactive_output(update_sallen_key, {'filter_type': filter_selector, 'R1': r1_slider, 'R2': r2_slider, 'R3': r3_slider, 'R4': r4_slider, 'C1': c1_slider, 'C2': c2_slider})

interactive_controls.layout.display = 'none'

# ------------------------------------------------------------
# 8. LEFT COLUMN
# ------------------------------------------------------------

control_column = VBox([legend_html, parameter_label, r1_slider, r2_slider, r3_slider, r4_slider, c1_slider, c2_slider, units_html], layout=Layout(width='230px', min_width='230px', flex='0 0 230px', align_items='flex-start', padding='2px 0px 0px 4px', overflow='hidden'))

# ------------------------------------------------------------
# 9. FILTER TYPE ROW
# ------------------------------------------------------------

filter_controls = HBox([filter_type_label, filter_selector], layout=Layout(width='100%', height='30px', align_items='center', justify_content='flex-start', column_gap='48px', margin='0px 0px 4px 0px', padding='0px'))

# ------------------------------------------------------------
# 10. RIGHT COLUMN
# ------------------------------------------------------------

right_column = VBox([filter_controls, magnitude_output, phase_output, info_output], layout=Layout(width='auto', min_width='0px', flex='1 1 auto', align_items='flex-start', overflow='hidden'))

# ------------------------------------------------------------
# 11. MAIN AREA
# ------------------------------------------------------------

main_area = HBox([control_column, right_column], layout=Layout(width='100%', max_width='100%', align_items='flex-start', justify_content='flex-start', overflow='hidden'))

# ------------------------------------------------------------
# 12. FINAL DISPLAY
# ------------------------------------------------------------

display(radio_style)
display(description)
display(main_area)
display(interactive_controls)